# ThermoRG Phase S1 v3 — Cooling Theory Validation (Compact)

**Goal**: Validate BatchNorm as a "cooling mechanism" (φ_BN ≈ 0.66)

**Design**:
- None_NoSkip: baseline (D=32,48,64,96 × 2 seeds = 8 runs)
  - D=32,48 × 2 seeds: **already completed** from v2 (4 runs)
  - D=64,96 × 2 seeds: **remaining** (4 runs)
- BN_NoSkip: test BatchNorm cooling (D=32,48,64,96 × 2 seeds = 8 runs)
- LN_NoSkip: control (D=32,48,64,96 × 1 seed = 4 runs)

**Total**: 16 runs × ~45min = **~12 hours** (within 17h quota)

**Checkpoint**: resume-safe, skips completed runs automatically

In [ ]:
import os, sys, subprocess, time, json
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from tqdm.auto import tqdm

## ▶️ Step 1: Clone Code from GitHub

In [ ]:
%cd /kaggle/working
!rm -rf ThermoRG-NN
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
gh_token = user_secrets.get_secret("gh_token")
repo_url = f"https://{gh_token}@github.com/xliu203/ThermoRG-NN.git"
!git clone {repo_url} --branch develop
%cd /kaggle/working/ThermoRG-NN
!git config --global user.email "xliu203@asu.edu"
!git config --global user.name "Leo Liu"
print("Cloned develop branch")

## ▶️ Step 2: Install Dependencies

In [ ]:
!pip install torch torchvision scipy numpy tqdm  # show output

import sys
sys.path.insert(0, "/kaggle/working/ThermoRG-NN")
sys.path.insert(0, "/kaggle/working/ThermoRG-NN/experiments/phase_s1")

import phase_s1_kaggle_v3 as p1

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch: {torch.__version__}")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## ▶️ Step 3: Smoke Test — Forward + Backward All 3 Configs

In [ ]:
x = torch.randn(4, 3, 32, 32)
y = torch.randint(0, 10, (4,))

SMOKE_CONFIGS = [
    ('None_NoSkip',  'none',       [42]),
    ('BN_NoSkip',    'batchnorm',  [42]),
    ('LN_NoSkip',    'layernorm',  [42]),
]

print("=== Smoke Test: Forward + Backward ===")
for cfg_name, norm_type, seeds in SMOKE_CONFIGS:
    model = p1.ValidationNet(base_ch=32, norm_type=norm_type, use_skip=False)
    out = model(x)
    loss = nn.functional.cross_entropy(out, y)
    loss.backward()
    ok = out.shape == (4, 10) and not torch.isnan(loss)
    print(f"  {cfg_name}: {'\u2705' if ok else '\u274c'} (loss={loss.item():.4f})")
    del model, out, loss

# J_topo test
model = p1.ValidationNet(base_ch=32, norm_type='none', use_skip=False)
weights = model.get_conv_weights()
J = p1.compute_J_topo(weights)
print(f"\nJ_topo test: {J:.4f} {'\u2705' if 0 < J <= 1 else '\u274c'}")

# Gamma tracking test
init_vars = p1.measure_init_variance(model, batch_size=32)
print(f"Variance tracker: {len(init_vars)} layers {'\u2705'}")
del model
print("\n\u2705 Smoke test passed!")

## ▶️ Step 4: Run Phase S1 v3 Training

**Resume**: Automatically skips completed runs via `metadata.json`.
**Remaining**: None(D=64,96×2), BN(D=32,48,64,96×2), LN(D=32,48,64,96×1) = 16 runs total.

In [ ]:
# Verify configs
print("Config summary:")
for cfg_name, norm_type, seeds in p1.CONFIGS:
    n_runs = len(seeds) * len(p1.D_VALUES)
    print(f"  {cfg_name}: {len(p1.D_VALUES)} D values \u00d7 {len(seeds)} seeds = {n_runs} runs")

total_runs = sum(len(seeds) * len(p1.D_VALUES) for _, _, seeds in p1.CONFIGS)
print(f"Total: {total_runs} runs")
print(f"Output dir: {p1.OUT_DIR}")
print(f"Checkpoint dir: {p1.CKPT_DIR}")

# Check existing metadata
meta_file = p1.OUT_DIR / 'metadata.json'
if meta_file.exists():
    with open(meta_file) as f:
        metadata = json.load(f)
    completed = [r for r in metadata['completed_runs'] if p1.is_valid_run(r)]
    print(f"Already completed: {len(completed)} runs")
    print(f"  Completed runs: {sorted(completed)}")
else:
    print("No existing metadata - starting fresh")

In [ ]:
from datetime import datetime

print("\n" + "="*70)
print("STARTING Phase S1 v3 Training")
print("="*70)

t_start = time.time()

# Run main training (handles checkpoint resume + skip completed)
all_results = p1.main()

total_time = (time.time() - t_start) / 60
print(f"\n{'='*70}")
print(f"ALL DONE! Total session time: {total_time:.1f} min")
print(f"{'='*70}")

## ▶️ Step 5: Push Results to GitHub

In [ ]:
OUT_DIR = "/kaggle/working/phase_s1_results_v3"
CKPT_DIR = f"{OUT_DIR}/checkpoints"
cmds = [
    f"git add {OUT_DIR}/*.json 2>/dev/null || true",
    f"git add {CKPT_DIR}/*.pt 2>/dev/null || true",
    f"git add experiments/phase_s1/notebooks/*.ipynb 2>/dev/null || true",
    'git commit -m "Data: Phase S1 v3 cooling theory validation from Kaggle" || true',
    'git push origin develop 2>&1 || true'
]

for cmd in cmds:
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True,
                         cwd="/kaggle/working/ThermoRG-NN")
    if res.stdout.strip():
        print(res.stdout.strip())

print("\n\u2705 Results pushed to GitHub!")